In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, zipfile
import pandas as pd

zip_path = '/content/drive/MyDrive/dataset.zip'
extract_path = '/content/dataset'
if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

data_root = os.path.join(extract_path, 'dataset')
csv_paths = glob.glob(os.path.join(data_root, '*', '*.csv'))
print(f"تعداد فایل‌های CSV: {len(csv_paths)}")

dfs = [pd.read_csv(p, encoding='utf-8-sig') for p in csv_paths]
df_all = pd.concat(dfs, ignore_index=True)

print(f"تعداد کل ردیف‌ها (اشعار): {len(df_all):,}")
print(f"تعداد شاعران یکتا: {df_all['poet'].nunique()}")

print("\n=== ستون‌های دیتاست و نوع داده ===")
print(df_all.dtypes)

print("\n=== نمونه‌ی یک ردیف کامل (اولین سطر) ===")
print(df_all.iloc[0].to_dict())

candidate_keywords = ['سبک', 'مکتب', 'style', 'school', 'era', 'period', 'دوره', 'century', 'قرن']
found_cols = [c for c in df_all.columns if any(k.lower() in str(c).lower() for k in candidate_keywords)]
print("\n=== ستون‌های مرتبط با سبک/مکتب/دوره (بر اساسِ نام) ===")
print(found_cols if found_cols else "هیچ ستونی با این نام‌ها پیدا نشد.")

for col in ['format', 'rhythm', 'rhyme']:
    if col in df_all.columns:
        print(f"\n--- ۱۰ مقدارِ پرتکرارِ ستونِ '{col}' ---")
        print(df_all[col].value_counts().head(10))
    else:
        print(f"\nستونِ '{col}' در دیتاست وجود ندارد.")

print("\n=== ۱۰ شاعرِ پرکارترین ===")
print(df_all['poet'].value_counts().head(10))
print("\n=== ۱۰ شاعرِ کم‌کارترین ===")
print(df_all['poet'].value_counts().tail(10))

In [ ]:
from collections import Counter

def get_positions(verses_str):
    lines = str(verses_str).strip().split('\n')
    return [line.split('|', 1)[0] for line in lines if '|' in line]

df_all['positions'] = df_all['verses'].fillna('').apply(get_positions)
df_all['is_all_neg1'] = df_all['positions'].apply(lambda ps: bool(ps) and all(p == '-1' for p in ps))
df_all['has_some_neg1'] = df_all['positions'].apply(lambda ps: ('-1' in ps) and not all(p == '-1' for p in ps))

pos_counter = Counter()
for ps in df_all['positions']:
    pos_counter.update(ps)

print(f"تعداد کل اشعار: {len(df_all):,}")
print(f"اشعاری که تماماً pos=-1 هستند (نثرِ خالص، بعد از پاک‌سازیِ خط‌لوله‌ی قبلی خالی می‌مانند): {df_all['is_all_neg1'].sum():,}")
print(f"اشعاری که ترکیبی از pos=-1 و بیتِ واقعی دارند: {df_all['has_some_neg1'].sum():,}")
print("\n۱۵ مقدارِ پرتکرارِ pos:")
print(pos_counter.most_common(15))

print("\nوضعیتِ format برای اشعاری که تماماً pos=-1 هستند:")
print(df_all[df_all['is_all_neg1']]['format'].isna().value_counts())

odd_formats = sorted({f for f in df_all['format'].dropna().unique() if str(f).startswith('قالب_')})
print(f"\nتعدادِ مقادیرِ format به‌شکلِ 'قالب_XXX': {len(odd_formats)}")
print(odd_formats)
print("\nتعدادِ اشعار در هر مقدارِ غیرعادی:")
print(df_all[df_all['format'].isin(odd_formats)]['format'].value_counts())

print(f"\nتعدادِ poem_id تکراری: {df_all['poem_id'].duplicated().sum()}")

print("\nآمارِ توصیفیِ verse_count:")
print(df_all['verse_count'].describe())

fully_missing = df_all[['format', 'rhythm', 'rhyme']].isna().all(axis=1)
print(f"\nتعدادِ اشعاری که هر سه‌یِ format/rhythm/rhyme را ندارند: {fully_missing.sum():,} از {len(df_all):,} ({fully_missing.mean()*100:.2f}%)")

In [ ]:
import zipfile
from collections import Counter

zip_path = '/content/drive/MyDrive/dataset.zip'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    all_files = zip_ref.namelist()

print(f"تعداد کل فایل‌های داخل zip: {len(all_files)}")

non_csv = [f for f in all_files if not f.lower().endswith('.csv')]
print(f"\nتعداد فایل‌های غیر-CSV: {len(non_csv)}")
print(non_csv[:40])

depths = Counter(f.count('/') for f in all_files)
print("\nتوزیع عمقِ مسیر فایل‌ها (تعداد '/'):")
print(depths)

root_level_files = [f for f in all_files if f.count('/') <= 1]
print(f"\nفایل‌های ریشه/یک‌سطحی: {len(root_level_files)}")
print(root_level_files[:40])

all_csv_in_zip = [f for f in all_files if f.lower().endswith('.csv')]
print(f"\nتعداد کل CSV در zip: {len(all_csv_in_zip)} | تعداد CSV بارگذاری‌شده با الگوی قبلی: {len(csv_paths)}")

from collections import defaultdict
per_folder = defaultdict(list)
for f in all_csv_in_zip:
    parts = f.split('/')
    if len(parts) >= 2:
        per_folder[parts[-2]].append(f)
multi_csv_folders = {k: v for k, v in per_folder.items() if len(v) > 1}
print(f"\nتعداد پوشه‌هایی که بیش از یک CSV دارند: {len(multi_csv_folders)}")
if multi_csv_folders:
    print(dict(list(multi_csv_folders.items())[:5]))

In [ ]:
import pandas as pd

raw_poet_counts = df_all['poet'].value_counts()
print("خام (پیش از پاک‌سازی):")
print("تعداد شاعران >= 50:", (raw_poet_counts >= 50).sum())
print()

def parse_and_clean_verses(verses_str, max_len=500):
    lines = verses_str.strip().split(chr(10))
    clean = []
    for line in lines:
        if '|' not in line:
            continue
        pos, text = line.split('|', 1)
        if pos == '-1' or len(text) > max_len:
            continue
        clean.append((pos, text))
    return clean

cleaned_col = []
for verses_str in df_all['verses']:
    cleaned_col.append(chr(10).join(f"{p}|{t}" for p, t in parse_and_clean_verses(verses_str)))

df_all_clean = df_all.copy()
df_all_clean['verses_clean'] = cleaned_col
before = len(df_all_clean)
df_all_clean = df_all_clean[df_all_clean['verses_clean'].str.len() > 0].reset_index(drop=True)
after = len(df_all_clean)
print(f"پاک‌سازی: {before:,} → {after:,} شعر (حذف {before - after:,} شعرِ کاملاً نثری)")
print()

clean_poet_counts = df_all_clean['poet'].value_counts()
print("بعد از پاک‌سازی:")
print("تعداد شاعران >= 50:", (clean_poet_counts >= 50).sum())
print()

compare = pd.DataFrame({'raw_count': raw_poet_counts, 'clean_count': clean_poet_counts}).fillna(0).astype(int)
compare['raw_keep'] = compare['raw_count'] >= 50
compare['clean_keep'] = compare['clean_count'] >= 50
changed = compare[compare['raw_keep'] != compare['clean_keep']]
print(f"شاعرانی که وضعیتِ نگه‌داشتن/حذف‌شدنشان تغییر کرد: {len(changed)}")
print(changed.sort_values('raw_count', ascending=False))

In [ ]:
MIN_POEMS_PER_POET = 50

clean_poet_counts = df_all_clean['poet'].value_counts()
eligible_poets = clean_poet_counts[clean_poet_counts >= MIN_POEMS_PER_POET].sort_values(ascending=False)

print(f"تعداد شاعران واجد شرایط (>= {MIN_POEMS_PER_POET} شعر، پس از پاک‌سازی): {len(eligible_poets)}")
print()
for name, count in eligible_poets.items():
    print(f"{name}: {count}")

poet_list = eligible_poets.index.tolist()
print()
print("--- PYTHON LIST (برای کپی مستقیم) ---")
print(poet_list)

In [ ]:
import json
from collections import Counter

poet_to_school = {
    "مسعود سعد سلمان": "خراسانی", "ابوسعید ابوالخیر": "خراسانی", "ابوالقاسم فردوسی": "خراسانی",
    "امیر معزی نیشابوری": "خراسانی", "رودکی سمرقندی": "خراسانی", "قطران تبریزی": "خراسانی",
    "سوزنی سمرقندی": "خراسانی", "عنصری بلخی": "خراسانی", "ایرانشان ابن ابی الخیر ": "خراسانی",
    "خواجه عبدالله انصاری": "خراسانی", "خیام نیشابوری": "خراسانی", "مهستی گنجوی": "خراسانی",
    "باباطاهر": "خراسانی", "فرخی سیستانی": "خراسانی", "ناصرخسرو قبادیانی": "خراسانی",
    "اسدی توسی": "خراسانی", "عثمان مختاری": "خراسانی", "لبیبی": "خراسانی",
    "کسایی مروزی": "خراسانی", "فخرالدین اسعد گرگانی": "خراسانی", "منوچهری دامغانی": "خراسانی",
    "عسجدی مروزی": "خراسانی", "شهید بلخی": "خراسانی", "عمعق بخاری": "خراسانی",
    "ابوالفرج رونی": "خراسانی", "سرایندهٔ فرامرزنامه": "خراسانی", "ازرقی هروی": "خراسانی",

    "جلال الدین محمد مولوی": "عراقی", "عطار نیشابوری": "عراقی", "جامی": "عراقی",
    "شاه نعمت\u200cالله ولی": "عراقی", "امیرخسرو دهلوی": "عراقی", "ابن یمین فریومدی": "عراقی",
    "سعدی شیرازی": "عراقی", "خواجوی کرمانی": "عراقی", "جهان ملک خاتون": "عراقی",
    "سنایی غزنوی": "عراقی", "اوحدالدین کرمانی": "عراقی", "حکیم نزاری قهستانی": "عراقی",
    "کمال\u200cالدین اسماعیل": "عراقی", "انوری ابیوردی": "عراقی", "خاقانی شروانی": "عراقی",
    "اوحدی مراغه\u200cای": "عراقی", "کمال خجندی": "عراقی", "سلمان ساوجی": "عراقی",
    "مجد همگر": "عراقی", "قاسم انوار": "عراقی", "حافظ شیرازی": "عراقی",
    "اسیری لاهیجی": "عراقی", "سیف فرغانی": "عراقی", "عراقی": "عراقی",
    "جمال\u200cالدین عبدالرزاق اصفهانی": "عراقی", "رشیدالدین میبدی": "عراقی",
    "مجیرالدین بیلقانی": "عراقی", "اثیر اخسیکتی": "عراقی", "سید حسن غزنوی": "عراقی",
    "ادیب صابر": "عراقی", "عمادالدین نسیمی": "عراقی", "جلال عضد": "عراقی",
    "همام تبریزی": "عراقی", "امیرعلیشیر نوایی": "عراقی", "نظامی گنجوی": "عراقی",
    "رشیدالدین وطواط": "عراقی", "ظهیر فاریابی": "عراقی", "ابن حسام خوسفی": "عراقی",
    "عبید زاکانی": "عراقی", "حسین خوارزمی": "عراقی", "شمس مغربی": "عراقی",
    "شیخ محمود شبستری": "عراقی", "کوهی": "عراقی", "باباافضل کاشانی": "عراقی",
    "نجم\u200cالدین رازی": "عراقی", "سلطان ولد": "عراقی", "عین\u200cالقضات همدانی": "عراقی",
    "امامی هروی": "عراقی", "قوامی رازی": "عراقی", "نصرالله منشی": "عراقی",
    "جنید شیرازی": "عراقی", "محمد بن منور": "عراقی", "سعدالدین وراوینی": "عراقی",
    "فلکی شروانی": "عراقی", "ظهیری سمرقندی": "عراقی", "فصیحی هروی": "عراقی",
    "خیالی بخارایی": "عراقی", "امیر شاهی سبزواری": "عراقی",
    "نظام قاری": "عراقی", "صوفی محمد هروی": "عراقی", "ابراهیم شاهدی دده": "عراقی",
    "سلیمی جرونی": "عراقی", "حیدر شیرازی": "عراقی", "عبدالقادر گیلانی": "عراقی",
    "سراج قمری": "عراقی", "محمد بن عمادالدین خراسانی": "عراقی",

    "صائب": "هندی", "بیدل دهلوی": "هندی", "اهلی شیرازی": "هندی", "حزین لاهیجی": "هندی",
    "جویای تبریزی": "هندی", "قدسی مشهدی": "هندی", "اسیر شهرستانی": "هندی",
    "فیض کاشانی": "هندی", "واعظ قزوینی": "هندی", "سلیم تهرانی": "هندی",
    "سیدای نسفی": "هندی", "محتشم کاشانی": "هندی", "فیاض لاهیجی": "هندی",
    "طغرای مشهدی": "هندی", "عرفی شیرازی": "هندی", "ناصر بخارایی": "هندی",
    "سعیدا": "هندی", "نظیری نیشابوری": "هندی", "بلند اقبال": "هندی", "کلیم": "هندی",
    "بابافغانی شیرازی": "هندی", "سام میرزا صفوی": "هندی", "وحشی بافقی": "هندی",
    "فضولی بغدادی": "هندی", "هلالی جغتایی": "هندی", "غالب دهلوی": "هندی",
    "قصاب کاشانی": "هندی", "شیخ بهایی": "هندی", "غنی کشمیری": "هندی",
    "ملا مسیح پانی\u200cپتی": "هندی", "وحیدالزمان قزوینی": "هندی", "میرداماد": "هندی",
    "امیر پازواری": "هندی", "ابوالحسن فراهانی": "هندی", "طغرل احراری": "هندی",
    "رضی\u200cالدین آرتیمانی": "هندی", "میلی": "هندی", "صابر همدانی": "هندی",
    "سلطان باهو": "هندی",

    "آشفتهٔ شیرازی": "بازگشت", "صغیر اصفهانی": "بازگشت", "صفایی جندقی": "بازگشت",
    "یغمای جندقی": "بازگشت", "قاآنی": "بازگشت", "مشتاق اصفهانی": "بازگشت",
    "آذر بیگدلی": "بازگشت", "فروغی بسطامی": "بازگشت", "نشاط اصفهانی": "بازگشت",
    "الهامی کرمانشاهی": "بازگشت", "ملا احمد نراقی": "بازگشت", "صامت بروجردی": "بازگشت",
    "سحاب اصفهانی": "بازگشت", "نورعلیشاه": "بازگشت", "ترکی شیرازی": "بازگشت",
    "رفیق اصفهانی": "بازگشت", "رضاقلی خان هدایت": "بازگشت", "نیر تبریزی": "بازگشت",
    "طبیب اصفهانی": "بازگشت", "صفای اصفهانی": "بازگشت", "قائم مقام فراهانی": "بازگشت",
    "حکیم سبزواری": "بازگشت", "هاتف اصفهانی": "بازگشت", "عمان سامانی": "بازگشت",
    "شاطر عباس صبوحی": "بازگشت", "محیط قمی": "بازگشت",
    "فایز": "بازگشت", "حاجب شیرازی": "بازگشت", "وفایی شوشتری": "بازگشت",
    "خالد نقشبندی": "بازگشت", "وفایی مهابادی": "بازگشت", "وحدت کرمانشاهی": "بازگشت",

    "صفی علیشاه": "معاصر", "ملک\u200cالشعرا بهار": "معاصر", "اقبال لاهوری": "معاصر",
    "ادیب الممالک فراهانی": "معاصر", "فرخی یزدی": "معاصر", "میرزا حبیب خراسانی": "معاصر",
    "محمدحسین غروی اصفهانی": "معاصر", "ایرج میرزا": "معاصر", "پروین اعتصامی": "معاصر",
    "شهریار": "معاصر", "عارف قزوینی": "معاصر", "رهی معیری": "معاصر",
    "میرزاده عشقی": "معاصر", "خلیل\u200cالله خلیلی": "معاصر", "جیحون یزدی": "معاصر",
    "میرزا آقاخان کرمانی": "معاصر", "افسر کرمانی": "معاصر", "غبار همدانی": "معاصر",
}

low_confidence = ["فایز", "امیر پازواری", "خالد نقشبندی", "وفایی مهابادی", "رضی\u200cالدین آرتیمانی"]

all_182 = list(df_all_clean["poet"].unique())
missing_in_dict = [p for p in all_182 if p not in poet_to_school]
extra_in_dict   = [p for p in poet_to_school if p not in all_182]
print("شاعرانی که در دیکشنری نیستند (باید خالی باشد):", missing_in_dict)
print("کلیدهای اضافه/اشتباه در دیکشنری (باید خالی باشد):", extra_in_dict)

print("\nتوزیع سبک‌ها:")
print(Counter(poet_to_school.values()))
print("\nتعداد کل:", len(poet_to_school))
print("تعداد شاعرانی که df_all_clean دارد:", len(all_182))

df_all_clean["school"] = df_all_clean["poet"].map(poet_to_school)

with open("poet_to_school.json", "w", encoding="utf-8") as f:
    json.dump(poet_to_school, f, ensure_ascii=False, indent=2)

with open("poets_low_confidence.json", "w", encoding="utf-8") as f:
    json.dump(low_confidence, f, ensure_ascii=False, indent=2)

print("\n✅ فایل‌ها ذخیره شدند: poet_to_school.json و poets_low_confidence.json")
print("✅ دیگر هیچ شاعری 'نامشخص' نیست — داده کامل شد.")

In [ ]:
import json
import random

with open("poet_to_school.json", "r", encoding="utf-8") as f:
    poet_to_school = json.load(f)

SEED = 42
TARGET_ZERO_SHOT = 32
EXCLUDE_FROM_ZERO_SHOT = ["صائب"]

school_to_poets = {}
for poet, school in poet_to_school.items():
    school_to_poets.setdefault(school, []).append(poet)

total_poets = len(poet_to_school)
schools_sorted = sorted(school_to_poets.keys(), key=lambda s: -len(school_to_poets[s]))

quota = {}
allocated = 0
for school in schools_sorted:
    n = len(school_to_poets[school])
    q = round(TARGET_ZERO_SHOT * n / total_poets)
    quota[school] = q
    allocated += q
diff = TARGET_ZERO_SHOT - allocated
quota[schools_sorted[0]] += diff

rng = random.Random(SEED)
zero_shot_poets = []
reserve_pools = {}
for school in schools_sorted:
    pool = sorted(school_to_poets[school])
    rng.shuffle(pool)
    picked = pool[: quota[school]]
    reserve_pools[school] = pool[quota[school]:]
    zero_shot_poets.extend(picked)
zero_shot_poets = sorted(zero_shot_poets)

for excluded_poet in EXCLUDE_FROM_ZERO_SHOT:
    if excluded_poet not in zero_shot_poets:
        print(f"⚠️ {excluded_poet} از قبل در Zero-Shot نبود.")
        continue
    school = poet_to_school[excluded_poet]
    replacement = next((c for c in reserve_pools[school] if c not in zero_shot_poets), None)
    if replacement is None:
        raise ValueError(f"جایگزین برای {excluded_poet} در مکتب {school} پیدا نشد")
    zero_shot_poets.remove(excluded_poet)
    zero_shot_poets.append(replacement)
    print(f"جایگزینی انجام شد: {excluded_poet} ({school}) ← {replacement} ({school})")

zero_shot_poets = sorted(zero_shot_poets)
closed_set_poets = sorted(set(poet_to_school.keys()) - set(zero_shot_poets))

print("\nسهمِ هر مکتب پس از جایگزینی:")
for school in schools_sorted:
    cnt = sum(1 for p in zero_shot_poets if poet_to_school[p] == school)
    print(f"  {school}: {cnt} از {len(school_to_poets[school])}")

print(f"\nجمع Zero-Shot: {len(zero_shot_poets)} | جمع Closed-Set: {len(closed_set_poets)} | کل: {len(zero_shot_poets)+len(closed_set_poets)}")

print("\nفهرست نهاییِ ۳۲ شاعر Zero-Shot:")
for p in zero_shot_poets:
    print(f"  {p} ({poet_to_school[p]})")

with open("zero_shot_poets.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "seed": SEED,
            "method": "stratified_random_by_school_proportional_with_manual_exclusions",
            "excluded_and_replaced": EXCLUDE_FROM_ZERO_SHOT,
            "zero_shot_poets": zero_shot_poets,
            "closed_set_poets": closed_set_poets,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )
print("\n✅ zero_shot_poets.json به‌روزرسانی و ذخیره شد.")

In [ ]:
print("ستون‌های df_all_clean:")
print(df_all_clean.columns.tolist())

print("\nنوعِ داده‌یِ ستونِ verses (یا verses_clean):")
verses_col = "verses_clean" if "verses_clean" in df_all_clean.columns else "verses"
print("استفاده از ستون:", verses_col)
print(type(df_all_clean[verses_col].iloc[0]))

print("\nنمونه‌یِ مقدارِ این ستون برایِ اولین شعر:")
sample_value = df_all_clean[verses_col].iloc[0]
print(sample_value if not hasattr(sample_value, "__len__") or isinstance(sample_value, str) else sample_value[:5])

print("\nنمونه‌ی کامل یک ردیف:")
print(df_all_clean.iloc[0])

In [ ]:
import pandas as pd

def parse_verses(verses_str):
    parsed = []
    for line in str(verses_str).split("\n"):
        if "|" not in line:
            continue
        pos_str, text = line.split("|", 1)
        try:
            pos = int(pos_str)
        except ValueError:
            continue
        parsed.append((pos, text))
    return parsed

records = []
for _, row in df_all_clean.iterrows():
    for pos, text in parse_verses(row["verses_clean"]):
        if pos in (2, 3, 4):
            records.append({
                "poet": row["poet"],
                "poem_id": row["poem_id"],
                "format": row["format"],
                "pos": pos,
                "text": text,
            })

anomalous_df = pd.DataFrame(records)

print("تعداد خطوط با pos غیرعادی (۲/۳/۴):", len(anomalous_df))
print("\nتوزیع pos:")
print(anomalous_df["pos"].value_counts())

print("\nتوزیع format برای این خطوط:")
print(anomalous_df["format"].value_counts())

print("\nتعداد شعرهای منحصربه‌فرد دارای این خطوط:", anomalous_df["poem_id"].nunique() if len(anomalous_df) else 0)

if len(anomalous_df):
    sample_poem_ids = anomalous_df["poem_id"].drop_duplicates().head(3)
    for pid in sample_poem_ids:
        poem_row = df_all_clean[df_all_clean["poem_id"] == pid].iloc[0]
        print(f"\n--- poem_id={pid} | poet={poem_row['poet']} | format={poem_row['format']} ---")
        for pos, text in parse_verses(poem_row["verses_clean"]):
            print(f"  pos={pos}: {text[:60]}")

In [ ]:
import pandas as pd
from collections import Counter

def parse_verses(verses_str):
    parsed = []
    for line in str(verses_str).split("\n"):
        if "|" not in line:
            continue
        pos_str, text = line.split("|", 1)
        try:
            pos = int(pos_str)
        except ValueError:
            continue
        parsed.append((pos, text))
    return parsed

def group_into_stanza_units(parsed_lines):
    units = []
    current = []
    for pos, text in parsed_lines:
        if pos == 0 and current:
            units.append(current)
            current = []
        current.append((pos, text))
    if current:
        units.append(current)
    return units

validation_issues = 0
unit_length_counter = Counter()
naive_pair_count = 0
stanza_unit_count = 0
affected_poems_with_issue = []

for _, row in df_all_clean.iterrows():
    parsed = parse_verses(row["verses_clean"])
    if not parsed:
        continue
    units = group_into_stanza_units(parsed)
    naive_pair_count += len(parsed) // 2
    stanza_unit_count += len(units)
    for unit in units:
        positions_in_unit = [p for p, _ in unit]
        expected = list(range(len(positions_in_unit)))
        if positions_in_unit != expected:
            validation_issues += 1
            if len(affected_poems_with_issue) < 5:
                affected_poems_with_issue.append((row["poem_id"], positions_in_unit))
        unit_length_counter[len(unit)] += 1

print("تعداد بندهای نامعتبر (توالیِ pos غیرمنتظره):", validation_issues)
print("نمونه‌ی چند موردِ نامعتبر (poem_id, positions):", affected_poems_with_issue)

print("\nتوزیعِ طولِ واحدها (تعداد خط در هر بند):")
print(dict(sorted(unit_length_counter.items())))

print(f"\nروش قدیم (جفت‌کردنِ ساده‌ی هر ۲ خط): {naive_pair_count} بیت")
print(f"روش جدید (گروه‌بندی بر اساسِ ریست‌شدنِ pos): {stanza_unit_count} واحد")

In [ ]:
import json
import random
from collections import Counter

random.seed(42)

def parse_verses(verses_clean_str):
    lines = []
    for line in str(verses_clean_str).split("\n"):
        if not line:
            continue
        pos_str, _, text = line.partition("|")
        try:
            pos = int(pos_str)
        except ValueError:
            continue
        lines.append((pos, text))
    return lines

def get_bait_units(verses_clean_str):
    parsed = parse_verses(verses_clean_str)
    units, current = [], []
    for pos, text in parsed:
        if pos == 0 and current:
            units.append(current)
            current = []
        current.append(text)
    if current:
        units.append(current)
    return ["\n".join(u) for u in units]

with open("zero_shot_poets.json", "r", encoding="utf-8") as f:
    zs_data = json.load(f)

zero_shot_poets = zs_data["zero_shot_poets"] if isinstance(zs_data, dict) else zs_data
closed_set_poets = zs_data.get("closed_set_poets") if isinstance(zs_data, dict) else None
if not closed_set_poets:
    closed_set_poets = sorted(set(df_all_clean["poet"].unique()) - set(zero_shot_poets))

print("Zero-Shot poets:", len(zero_shot_poets))
print("Closed-Set poets:", len(closed_set_poets))

zs_mask = df_all_clean["poet"].isin(zero_shot_poets)
df_zero_shot = df_all_clean[zs_mask].copy()
df_closed = df_all_clean[~zs_mask].copy()

print("Zero-Shot poems:", len(df_zero_shot))
print("Closed-Set poems:", len(df_closed))

TRAIN_FRAC, VAL_FRAC = 0.8, 0.1

train_ids, val_ids, test_ids = [], [], []
for poet, group in df_closed.groupby("poet"):
    ids = group["poem_id"].tolist()
    rng = random.Random(42)
    rng.shuffle(ids)
    n = len(ids)
    n_train = min(int(round(n * TRAIN_FRAC)), n)
    n_val = min(int(round(n * VAL_FRAC)), n - n_train)
    train_ids.extend(ids[:n_train])
    val_ids.extend(ids[n_train:n_train + n_val])
    test_ids.extend(ids[n_train + n_val:])

train_ids, val_ids, test_ids = set(train_ids), set(val_ids), set(test_ids)
assert train_ids.isdisjoint(val_ids) and train_ids.isdisjoint(test_ids) and val_ids.isdisjoint(test_ids)
assert len(train_ids) + len(val_ids) + len(test_ids) == len(df_closed)

df_train = df_closed[df_closed["poem_id"].isin(train_ids)].copy()
df_val = df_closed[df_closed["poem_id"].isin(val_ids)].copy()
df_test_closed = df_closed[df_closed["poem_id"].isin(test_ids)].copy()
df_test_open = df_zero_shot.copy()

print("Train poems:", len(df_train))
print("Val poems:", len(df_val))
print("Test-Closed poems:", len(df_test_closed))
print("Test-Open poems:", len(df_test_open))

missing_from_train = set(df_closed["poet"].unique()) - set(df_train["poet"].unique())
print("شاعرانِ Closed-Set که در train دیده نمی‌شوند (باید خالی یا نزدیک‌به‌خالی باشد):", missing_from_train)

def build_full_records(df_subset, split_name):
    records = []
    for _, row in df_subset.iterrows():
        units = get_bait_units(row["verses_clean"])
        records.append({
            "poem_id": row["poem_id"],
            "poet": row["poet"],
            "school": row.get("school"),
            "format": row["format"],
            "rhythm": row["rhythm"],
            "rhyme": row["rhyme"],
            "num_baits": len(units),
            "text": "\n".join(units),
            "split": split_name,
        })
    return records

train_records = build_full_records(df_train, "train")
val_records = build_full_records(df_val, "val")
print("Train samples:", len(train_records))
print("Val samples:", len(val_records))

CHUNK_LENGTHS = [1, 2, 4, 6]

def build_test_records(df_subset, split_name):
    records = []
    for _, row in df_subset.iterrows():
        units = get_bait_units(row["verses_clean"])
        n_units = len(units)
        for k in CHUNK_LENGTHS:
            for start in range(0, n_units - k + 1, k):
                records.append({
                    "poem_id": row["poem_id"],
                    "poet": row["poet"],
                    "school": row.get("school"),
                    "format": row["format"],
                    "rhythm": row["rhythm"],
                    "rhyme": row["rhyme"],
                    "k": k,
                    "start_unit_idx": start,
                    "text": "\n".join(units[start:start + k]),
                    "split": split_name,
                })
    return records

test_closed_records = build_test_records(df_test_closed, "test_closed")
test_open_records = build_test_records(df_test_open, "test_open")

print("Test-Closed samples (همه k):", len(test_closed_records), "| by k:", Counter(r["k"] for r in test_closed_records))
print("Test-Open samples (همه k):", len(test_open_records), "| by k:", Counter(r["k"] for r in test_open_records))

def write_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

write_jsonl(train_records, "train.jsonl")
write_jsonl(val_records, "val.jsonl")
write_jsonl(test_closed_records, "test_closed.jsonl")
write_jsonl(test_open_records, "test_open.jsonl")
print("هر ۴ فایلِ JSONL نوشته شدند: train.jsonl, val.jsonl, test_closed.jsonl, test_open.jsonl")

In [ ]:
train_poem_ids = set(r["poem_id"] for r in train_records)
val_poem_ids = set(r["poem_id"] for r in val_records)
test_closed_poem_ids = set(r["poem_id"] for r in test_closed_records)
test_open_poem_ids = set(r["poem_id"] for r in test_open_records)

pairs = [
    ("train", train_poem_ids, "val", val_poem_ids),
    ("train", train_poem_ids, "test_closed", test_closed_poem_ids),
    ("train", train_poem_ids, "test_open", test_open_poem_ids),
    ("val", val_poem_ids, "test_closed", test_closed_poem_ids),
    ("val", val_poem_ids, "test_open", test_open_poem_ids),
    ("test_closed", test_closed_poem_ids, "test_open", test_open_poem_ids),
]

print("=== بررسیِ نشتِ داده (هم‌پوشانیِ poem_id) ===")
leak_found = False
for name_a, set_a, name_b, set_b in pairs:
    overlap = set_a & set_b
    print(f"{name_a} ∩ {name_b}: {len(overlap)} {'❌ نشت!' if overlap else '✅ بدون نشت'}")
    if overlap:
        leak_found = True

print("\n=== بررسیِ شاعرانِ Zero-Shot (نباید هرگز در train/val دیده شوند) ===")
zs_leak = (set(r["poet"] for r in train_records) | set(r["poet"] for r in val_records)) & set(zero_shot_poets)
print("شاعرانِ Zero-Shot در train/val:", zs_leak if zs_leak else "خالی ✅")

print("\nنتیجه‌ی کلی:", "❌ نشتِ داده پیدا شد!" if (leak_found or zs_leak) else "✅ بدون هیچ نشتی — همه‌چیز تأیید شد.")

In [ ]:
import os, shutil

PROJECT_DIR = "/content/drive/MyDrive/PACL_Project/phase1_outputs"
os.makedirs(PROJECT_DIR, exist_ok=True)

files_to_save = [
    "train.jsonl",
    "val.jsonl",
    "test_closed.jsonl",
    "test_open.jsonl",
    "zero_shot_poets.json",
    "poet_to_school.json",
    "poets_low_confidence.json",
]

saved, missing = [], []
for fname in files_to_save:
    if os.path.exists(fname):
        shutil.copy(fname, os.path.join(PROJECT_DIR, fname))
        saved.append(fname)
    else:
        missing.append(fname)

print("ذخیره‌شده در گوگل‌درایو:", saved)
print("پیدا نشد (باید بررسی شود چرا):", missing)

print("\nمحتویاتِ نهاییِ پوشه:")
for f in sorted(os.listdir(PROJECT_DIR)):
    size_kb = os.path.getsize(os.path.join(PROJECT_DIR, f)) / 1024
    print(f"  {f}  ({size_kb:.1f} KB)")

In [ ]:
import os, json

PROJECT_DIR = "/content/drive/MyDrive/PACL_Project/phase1_outputs"

df_all_clean.to_pickle(os.path.join(PROJECT_DIR, "df_all_clean.pkl"))

manifest = {
    "phase": 1,
    "canonical_poet_count": 182,
    "zero_shot_poet_count": len(zero_shot_poets),
    "closed_set_poet_count": len(closed_set_poets),
    "bait_unit_definition": "group lines by pos-reset-to-0 (NOT naive 2-line pairing)",
    "chunking_scope": "train/val = full poem text, no fixed-length chunking; test_closed/test_open = exact k in {1,2,4,6} bait-unit windows, non-overlapping",
    "splits": {
        "train": {"poems": len(df_train), "jsonl_samples": len(train_records)},
        "val": {"poems": len(df_val), "jsonl_samples": len(val_records)},
        "test_closed": {"poems": len(df_test_closed), "jsonl_samples": len(test_closed_records)},
        "test_open": {"poems": len(df_test_open), "jsonl_samples": len(test_open_records)},
    },
    "files": [
        "train.jsonl", "val.jsonl", "test_closed.jsonl", "test_open.jsonl",
        "zero_shot_poets.json", "poet_to_school.json", "poets_low_confidence.json",
        "df_all_clean.pkl",
    ],
    "leakage_validation": "zero poem_id overlap across all 4 splits; zero zero-shot poets in train/val (confirmed)",
}

with open(os.path.join(PROJECT_DIR, "phase1_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("محتویاتِ نهاییِ پوشه‌ی فاز ۱:")
for fname in sorted(os.listdir(PROJECT_DIR)):
    size_kb = os.path.getsize(os.path.join(PROJECT_DIR, fname)) / 1024
    print(f"  {fname}  ({size_kb:.1f} KB)")